# Trainer: TabNet
Run this entirely on its own Colab Tab!


In [ ]:
!pip install optuna pytorch-tabnet


In [ ]:
import pandas as pd
import numpy as np
import optuna
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
import torch


In [ ]:
POWER_5_SCHOOLS = {
    'Alabama', 'LSU', 'Georgia', 'Florida', 'Auburn', 'Arkansas', 'Texas A&M', 'Missouri', 
    'South Carolina', 'Ole Miss', 'Mississippi St.', 'Tennessee', 'Kentucky', 'Vanderbilt', 'Mississippi',
    'Ohio St.', 'Penn St.', 'Wisconsin', 'Michigan', 'Michigan St.', 'Iowa', 'Nebraska', 
    'Minnesota', 'Illinois', 'Northwestern', 'Purdue', 'Indiana', 'Rutgers', 'Maryland',
    'Clemson', 'Florida St.', 'Miami (FL)', 'Virginia Tech', 'North Carolina', 'Duke', 
    'Georgia Tech', 'Louisville', 'NC State', 'Pittsburgh', 'Boston College', 'Syracuse', 
    'Virginia', 'Wake Forest', 'Miami',
    'Oklahoma', 'Texas', 'West Virginia', 'TCU', 'Baylor', 'Oklahoma St.', 'Kansas St.', 
    'Iowa St.', 'Texas Tech', 'Kansas', 'BYU', 'Houston', 'UCF', 'Cincinnati',
    'USC', 'UCLA', 'Stanford', 'Oregon', 'Washington', 'Utah', 'Arizona St.', 'California', 
    'Washington St.', 'Oregon St.', 'Colorado', 'Arizona', 'Notre Dame'
}

def get_archetype(row):
    h, w = row['Height'], row['Weight']
    if pd.isnull(h) or pd.isnull(w): return 'Lightweight_Skills'
    if w >= 120: return 'Heavyweight_Lineman'
    elif w >= 95: return 'Big_Hybrid_Edge_TE' if h >= 1.90 else 'Dense_Power_LB_RB'
    else: return 'Tall_Speed_WR_DB' if h >= 1.88 else 'Lightweight_Skills'

def engineer_features(df):
    df = df.copy()
    df['Power_Factor'] = df['Weight'] * df['Vertical_Jump']
    df['Speed_to_Size_Ratio'] = df['Sprint_40yd'] / df['Weight']
    df['Total_Jump'] = df['Vertical_Jump'] + df['Broad_Jump']
    df['Agility_Score'] = df['Agility_3cone'] + df['Shuttle']
    df['BMI'] = (df['Weight'] / (df['Height'] ** 2)) * 703
    df['Momentum'] = df['Weight'] * (40.0 / df['Sprint_40yd'])
    df['Jump_Power_Index'] = df['Weight'] * df['Total_Jump']
    df['Agility_Speed_Ratio'] = df['Agility_Score'] / df['Sprint_40yd']
    df['School_Conference'] = df['School'].apply(lambda x: 'Power_5' if x in POWER_5_SCHOOLS else 'Other_Conference')
    df['Physical_Archetype'] = df.apply(get_archetype, axis=1)
    return df


In [ ]:
# Make sure to upload train.csv and test.csv directly to Colab's file explorer!
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train = engineer_features(train)
test = engineer_features(test)

X = train.drop(['Drafted', 'Id'], axis=1)
y = train['Drafted']
X_test_raw = test.drop(['Id'], axis=1)

cat_cols = ['School', 'Position', 'Player_Type', 'Position_Type', 'Year', 'School_Conference', 'Physical_Archetype']
num_cols_to_clip = ['Age', 'Height', 'Weight', 'Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle', 'BMI', 'Speed_Score', 'Power_Factor', 'Speed_to_Size_Ratio', 'Total_Jump', 'Agility_Score', 'Explosiveness_Index', 'Bench_to_Weight_Ratio', 'Speed_to_Weight_Efficiency', 'Explosiveness_to_Weight_Efficiency', 'Momentum', 'Jump_Power_Index', 'Agility_Speed_Ratio']


In [ ]:
class TabNetPreprocessor:
    def __init__(self, cat_cols):
        self.cat_cols = cat_cols
        self.encoders = {}
        self.cat_dims = []
        self.cat_idxs = []
        self.num_imputer = SimpleImputer(strategy='median')
        self.num_scaler = StandardScaler()

    def fit_transform(self, X, y=None):
        X = X.copy()
        self.cat_idxs = [X.columns.get_loc(c) for c in self.cat_cols if c in X.columns]
        num_cols = [c for c in X.columns if c not in self.cat_cols]
        X[num_cols] = self.num_imputer.fit_transform(X[num_cols])
        X[num_cols] = self.num_scaler.fit_transform(X[num_cols])
        
        for col in self.cat_cols:
            if col in X.columns:
                X[col] = X[col].astype(str).fillna('missing')
                le = LabelEncoder()
                X[col] = le.fit_transform(X[col])
                self.encoders[col] = le
                self.cat_dims.append(len(le.classes_) + 1)
        return X

    def transform(self, X):
        X = X.copy()
        num_cols = [c for c in X.columns if c not in self.cat_cols]
        X[num_cols] = self.num_imputer.transform(X[num_cols])
        X[num_cols] = self.num_scaler.transform(X[num_cols])
        
        for col in self.cat_cols:
            if col in X.columns:
                X[col] = X[col].astype(str).fillna('missing')
                le = self.encoders[col]
                unknown_class = len(le.classes_)
                X[col] = X[col].apply(lambda x: np.where(le.classes_ == x)[0][0] if x in le.classes_ else unknown_class)
        return X


In [ ]:
import torch
from pytorch_tabnet.tab_model import TabNetClassifier

def train_and_validate(X, y, params):
    skf = StratifiedKFold(n_splits=5, shuffle=False)
    scores = []
    oof_probs = np.zeros(len(X))
    fitted_models = []
    
    for train_idx, val_idx in skf.split(X, y):
        X_train_raw, X_val_raw = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        pipeline = TabNetPreprocessor(cat_cols)
        X_train = pipeline.fit_transform(X_train_raw).values
        X_val = pipeline.transform(X_val_raw).values
        
        model = TabNetClassifier(**params, cat_idxs=pipeline.cat_idxs, cat_dims=pipeline.cat_dims, verbose=0)
        model.fit(
            X_train, y_train.values,
            eval_set=[(X_val, y_val.values)],
            eval_metric=['auc'],
            max_epochs=100,
            patience=20,
            batch_size=256,
            virtual_batch_size=128
        )
        probs = model.predict_proba(X_val)[:, 1]
        
        oof_probs[val_idx] = probs
        scores.append(roc_auc_score(y_val, probs))
        fitted_models.append((pipeline, model))
        
    return np.mean(scores), oof_probs, fitted_models

def objective(trial):
    n_d = trial.suggest_int('n_d', 8, 64)
    params = {
        'n_d': n_d,
        'n_a': n_d,
        'n_steps': trial.suggest_int('n_steps', 3, 10),
        'gamma': trial.suggest_float('gamma', 1.0, 2.0),
        'n_independent': trial.suggest_int('n_independent', 1, 5),
        'n_shared': trial.suggest_int('n_shared', 1, 5),
        'momentum': trial.suggest_float('momentum', 0.01, 0.4)
    }
    score, _, _ = train_and_validate(X, y, params)
    return score


In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

print("Best Score:", study.best_value)
print("Best Params:", study.best_params)

_, oof_probs, models = train_and_validate(X, y, study.best_params)

test_probs = np.zeros(len(X_test_raw))
for pipeline, model in models:
    X_test_processed = pipeline.transform(X_test_raw).values
    test_probs += model.predict_proba(X_test_processed)[:, 1] / len(models)

pd.Series(oof_probs).to_csv('tabnet_oof.csv', index=False)
pd.Series(test_probs).to_csv('tabnet_test.csv', index=False)
print("Saved TabNet predictions to tabnet_oof.csv and tabnet_test.csv locally in Colab. Please download them!")
